# ADAS-Vision — Google Colab T4 Full Pipeline (A → Z)

Notebook tổng hợp chạy trọn vẹn trên **Google Colab (T4 GPU)**:
1. Kiểm tra GPU & Mount Google Drive
2. Clone/mở dự án & cài đặt thư viện
3. Giải nén dữ liệu gốc BDD100K & Cityscapes vào SSD `/content/datasets` (tốc độ cao)
4. Khảo sát nhãn thật (`inspect_bdd.py`) & tạo split + nhãn YOLO (`prepare_data.py`)
5. Huấn luyện Object Detection **YOLO11** (`detector_best.pt`)
6. Huấn luyện Semantic Segmentation **DeepLabV3+** (`segmenter_best.pt`)
7. Đánh giá định lượng (`evaluate.py`: mAP, IoU, Dice, Cityscapes, FPS 720p) & Xuất Video Demo HUD (`run_inference.py`)

## Cell 1 — Kiểm tra GPU T4 & Kết nối Google Drive

In [ ]:
!nvidia-smi

from google.colab import drive

drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/adas_outputs/weights
!mkdir -p /content/drive/MyDrive/adas_outputs/reports
!mkdir -p /content/drive/MyDrive/adas_outputs/videos

## Cell 2 — Tải Mã Nguồn Dự Án & Cài Đặt Thư Viện
Điền URL GitHub repo của nhóm (hoặc bỏ comment dòng `%cd` nếu để code trực tiếp trên Google Drive).

In [ ]:
# Cách A: Clone từ GitHub (thay URL repo của nhóm)
# !git clone https://github.com/<username>/<repo-name>.git /content/adas-vision
# %cd /content/adas-vision

# Cách B: Hoặc trỏ vào thư mục code trên Google Drive
%cd /content/drive/MyDrive/Developing-utility-vehicles-and-regional-transit-networks

!pip install -q ultralytics segmentation-models-pytorch albumentations pyyaml cityscapesscripts

## Cell 3 — Giải Nén Dataset Gốc Sang Ổ SSD `/content/datasets`
Đặt sẵn các file `.zip` gốc trong `MyDrive/adas_data/` trên Google Drive để giải nén nhanh vào SSD nội bộ của Colab.

In [ ]:
import zipfile
from pathlib import Path

drive_data = Path('/content/drive/MyDrive/adas_data')
bdd_dest = Path('/content/datasets/bdd100k')
bdd_dest.mkdir(parents=True, exist_ok=True)

bdd_zips = [
    '100k_images_train.zip',
    '100k_images_val.zip',
    'bdd100k_det_20_labels_trainval.zip',
    'bdd100k_drivable_labels_trainval.zip',
    'bdd100k_lane_labels_trainval.zip',
]
for name in bdd_zips:
    zpath = drive_data / name
    if zpath.exists():
        print(f'Unpacking {name}...')
        with zipfile.ZipFile(zpath, 'r') as zf:
            zf.extractall(bdd_dest)
    else:
        print(f'[Skip] Chưa thấy file: {zpath}')

## Cell 4 — Khảo Sát Dữ Liệu Thật & Chuẩn Bị Nhãn / Splits (`seed=42`)

In [ ]:
import os

os.environ["BDD_ROOT"] = "/content/datasets/bdd100k"
os.environ["CITYSCAPES_ROOT"] = "/content/datasets/cityscapes"
os.environ["DEVICE"] = "cuda"

# 1. Kiểm tra thực tế cấu trúc thư mục, JSON categories và pixel ID của mask
!python scripts/inspect_bdd.py \
    --save-json /content/drive/MyDrive/adas_outputs/reports/bdd_inspection.json

# 2. Sinh split cố định (seed=42) và nhãn .txt cho YOLO11
# Gợi ý: 20,000 train / 2,000 val chạy vừa khít phiên Colab T4 miễn phí.
# Bỏ --max-train và --max-val nếu muốn chạy toàn bộ 70,000 train / 10,000 val.
!python scripts/prepare_data.py --max-train 20000 --max-val 2000

## Cell 5 — Huấn Luyện Object Detection (YOLO11) & Benchmark

In [ ]:
# Huấn luyện mô hình chính YOLO11s -> lưu trực tiếp sang Google Drive
!python scripts/train_detector.py \
    --data configs/bdd_yolo.yaml \
    --model yolo11s.pt \
    --epochs 30 \
    --batch-size 16 \
    --device 0 \
    --output-weights /content/drive/MyDrive/adas_outputs/weights/detector_best.pt

!cp /content/drive/MyDrive/adas_outputs/weights/detector_best.pt weights/detector_best.pt

# (Tùy chọn) Huấn luyện thêm YOLO11n / YOLOv8s để lập bảng đối sánh Benchmark:
# !python scripts/train_detector.py --data configs/bdd_yolo.yaml --model yolo11n.pt \
#     --epochs 30 --batch-size 32 --device 0 \
#     --output-weights /content/drive/MyDrive/adas_outputs/weights/yolo11n_best.pt

## Cell 6 — Huấn Luyện Semantic Segmentation (DeepLabV3+)

In [ ]:
# Nếu kết quả inspect_bdd.py ở Cell 4 báo pixel ID khác mặc định (0, 1, 255),
# hãy truyền thêm --direct-id <id> --alt-id <id> --lane-bg-id <id>
!python scripts/train_segmenter.py \
    --epochs 25 \
    --batch-size 8 \
    --device cuda \
    --output-weights /content/drive/MyDrive/adas_outputs/weights/segmenter_best.pt

!cp /content/drive/MyDrive/adas_outputs/weights/segmenter_best.pt weights/segmenter_best.pt

## Cell 7 — Đánh Giá Toàn Diện (BDD100K + Cityscapes + FPS T4) & Xuất Video Demo

In [ ]:
# Khôi phục weights từ Drive (hữu ích khi mở phiên Colab mới chỉ để đánh giá/xuất video)
!cp /content/drive/MyDrive/adas_outputs/weights/detector_best.pt weights/detector_best.pt
!cp /content/drive/MyDrive/adas_outputs/weights/segmenter_best.pt weights/segmenter_best.pt

# 1. Xuất báo cáo thực nghiệm định lượng (JSON + bảng Markdown)
!python scripts/evaluate.py \
    --mode all \
    --device cuda \
    --output-json /content/drive/MyDrive/adas_outputs/reports/eval_report.json \
    --output-md /content/drive/MyDrive/adas_outputs/reports/evaluation_report.md

# Hiển thị ngay bảng kết quả Markdown vừa đo được
!cat /content/drive/MyDrive/adas_outputs/reports/evaluation_report.md

# 2. Xuất video demo có vẽ HUD cảnh báo ADAS (đặt sẵn clip .mp4 vào MyDrive/adas_data/)
# !python scripts/run_inference.py run \
#     --source /content/drive/MyDrive/adas_data/day_clear.mp4 \
#     --output /content/drive/MyDrive/adas_outputs/videos/demo_day_out.mp4 \
#     --log-csv /content/drive/MyDrive/adas_outputs/videos/events_day.csv \
#     --device cuda \
#     --no-display